# Sprint B — Step 0 Pre-flight

Objectif : verifier les assets Sprint A et entrainer les modeles LightGBM quantile manquants.

Sorties :
- `models/lgbm_quantile_p10.pkl`, `_p50.pkl`, `_p90.pkl`
- `data/processed/stock_mock.parquet`
- Validation pipeline `predict_qte_v2`

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error

from dashboard.utils.features import FEATURES_V2, TARGET_QTE
from dashboard.utils.stock_mock import generate_stock_mock, compute_adu, compute_lt

DATA = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'
MODELS.mkdir(exist_ok=True)
print('ROOT:', ROOT)

ROOT: c:\Users\lenovo\Desktop\Extraction livraison client 2021-2025


## 1. Chargement des splits v3 + verification features

In [2]:
df_train = pd.read_parquet(DATA / 'split_train_v3_features.parquet')
df_val = pd.read_parquet(DATA / 'split_val_v3_features.parquet')
df_test = pd.read_parquet(DATA / 'split_test_v3_features.parquet')
print('Train:', df_train.shape, '| Val:', df_val.shape, '| Test:', df_test.shape)
missing = [f for f in FEATURES_V2 if f not in df_train.columns]
assert not missing, f'Missing features: {missing}'
print(f'Toutes les {len(FEATURES_V2)} features V2 presentes dans les splits.')

Train: (210641, 71) | Val: (70871, 71) | Test: (66174, 71)
Toutes les 47 features V2 presentes dans les splits.


## 2. Validation XGBoost v2 (sanity check)

In [3]:
xgb_v2 = joblib.load(MODELS / 'xgboost_optuna_v2.pkl')
y_pred_log = xgb_v2.predict(df_test[FEATURES_V2])
y_pred = np.clip(np.expm1(y_pred_log), 0, None)
y_true = df_test[TARGET_QTE].values
mae_v2 = mean_absolute_error(y_true, y_pred)
print(f'XGBoost v2 MAE test = {mae_v2:.3f} (Sprint A reporte 8.42)')
assert mae_v2 < 11, 'MAE v2 > 11 : modele probablement corrompu'

XGBoost v2 MAE test = 8.422 (Sprint A reporte 8.42)


## 3. Stock mock generator

In [4]:
stock = generate_stock_mock(df_train, coverage_days=30, seed=42)
stock.to_parquet(DATA / 'stock_mock.parquet')
print('Stock mock :', stock.shape)
stock.head()

Stock mock : (359, 5)


,code_article_freq,on_hand,on_order,adu,lt_jours
0,1,19.79,0.0,0.5667,8.5
1,2,28.26,0.0,0.9778,5.0
2,3,53.47,0.0,1.4667,6.0
3,4,19.76,0.0,0.5889,7.0
4,5,13.62,3.0,0.6000,5.0


## 4. Entrainement LightGBM Quantile (P10, P50, P90)

Hyperparams legers (objectif : disposer d'un intervalle decent, pas d'un modele prod-grade).

In [5]:
import lightgbm as lgb

y_train_log = np.log1p(df_train[TARGET_QTE].clip(lower=0))
y_val_log = np.log1p(df_val[TARGET_QTE].clip(lower=0))
X_train = df_train[FEATURES_V2]
X_val = df_val[FEATURES_V2]
X_test = df_test[FEATURES_V2]

params_common = dict(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    feature_fraction=0.9,
    bagging_fraction=0.9,
    bagging_freq=5,
    verbosity=-1,
    objective='quantile',
)

models_q = {}
for q, alpha in [('p10', 0.1), ('p50', 0.5), ('p90', 0.9)]:
    m = lgb.LGBMRegressor(alpha=alpha, **params_common)
    m.fit(X_train, y_train_log, eval_set=[(X_val, y_val_log)], callbacks=[lgb.early_stopping(30)])
    joblib.dump(m, MODELS / f'lgbm_quantile_{q}.pkl')
    models_q[q] = m
    print(f'{q} (alpha={alpha}) sauvegarde.')

Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[400]	valid_0's quantile: 0.0598492
p10 (alpha=0.1) sauvegarde.
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[400]	valid_0's quantile: 0.1528
p50 (alpha=0.5) sauvegarde.
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[400]	valid_0's quantile: 0.0812052
p90 (alpha=0.9) sauvegarde.


## 5. Verification couverture de l'intervalle [P10, P90] sur test

In [6]:
pred_p10 = np.clip(np.expm1(models_q['p10'].predict(X_test)), 0, None)
pred_p50 = np.clip(np.expm1(models_q['p50'].predict(X_test)), 0, None)
pred_p90 = np.clip(np.expm1(models_q['p90'].predict(X_test)), 0, None)
coverage = ((y_true >= pred_p10) & (y_true <= pred_p90)).mean()
print(f'Couverture empirique intervalle [P10, P90] : {coverage:.1%} (cible nominale 80%)')
print(f'P50 MAE : {mean_absolute_error(y_true, pred_p50):.3f}')
print(f'Largeur moyenne intervalle : {(pred_p90 - pred_p10).mean():.2f}')

Couverture empirique intervalle [P10, P90] : 74.6% (cible nominale 80%)
P50 MAE : 8.323
Largeur moyenne intervalle : 25.15


## 6. Resume Step 0

- LGBM quantile entraine et sauvegarde
- Stock mock genere et persiste
- Pipeline XGBoost v2 valide

Pre-flight termine. Les chantiers parallelisables (C3, C4, C6) peuvent demarrer.